In [1]:
import pandas as pd

TARGET_COL = "TARGET"

df = pd.read_csv(r"D:\KL-lieuvth2\master_data_train_test_split.csv").iloc[:, 1:]

# Thêm Overall
d = pd.concat([
    df.assign(flag_train_test="Overall"),
    df
])

result = (
    d.groupby(["flag_train_test", TARGET_COL])
     .size()
     .unstack(fill_value=0)
     .rename(columns={0: "Good", 1: "Bad"})
)

result["Total"] = result["Good"] + result["Bad"]
result["Good %"] = result["Good"] / result["Total"] * 100
result["Bad %"]  = result["Bad"] / result["Total"] * 100

result = result[["Good", "Bad", "Total", "Good %", "Bad %"]]

print(result.round(2))

TARGET             Good    Bad   Total  Good %  Bad %
flag_train_test                                      
Overall          282686  24825  307511   91.93   8.07
test              84806   7448   92254   91.93   8.07
train            197880  17377  215257   91.93   8.07


In [3]:
import pandas as pd

# =========================
# Config
# =========================
FILE = r"D:\KL-lieuvth2\master_data_train_test_split.csv"

ID_COL      = "SK_ID_CURR"
TARGET_COL  = "TARGET"
DATASET_COL = "flag_train_test"

NUMERIC_OUT = r"D:\KL-lieuvth2\numeric_statistics.csv"
CATE_OUT    = r"D:\KL-lieuvth2\categorical_statistics.csv"

# =========================
# Load data
# =========================
df = pd.read_csv(FILE).iloc[:, 1:]

features = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL, DATASET_COL]
]

# =========================
# Phân loại feature
# =========================
numeric_features = df[features].select_dtypes(include="number").columns.tolist()

# Numeric chỉ có 0/1 -> treat như categorical
flag_features = [
    c for c in numeric_features
    if set(df[c].dropna().unique()).issubset({0, 1})
]

# Numeric thực sự
num_features = [
    c for c in numeric_features
    if c not in flag_features
]

# Non-numeric + binary flag
cat_features = (
    df[features].select_dtypes(exclude="number").columns.tolist()
    + flag_features
)

# =========================
# 1. Numeric statistics
# =========================
numeric_stats = (
    df[num_features]
    .describe()
    .T
    .rename(columns={
        "count": "Count",
        "mean": "Mean",
        "std": "Std",
        "min": "Min",
        "25%": "Q1",
        "50%": "Median",
        "75%": "Q3",
        "max": "Max"
    })
    .reset_index()
    .rename(columns={"index": "Feature"})
)

numeric_stats.to_csv(NUMERIC_OUT, index=False)

# =========================
# 2. Categorical statistics
# =========================
cate_list = []

for col in cat_features:
    stats = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("Value")
        .reset_index(name="Count")
    )

    stats["Feature"] = col
    stats["Percent"] = stats["Count"] / len(df) * 100

    # Đưa Feature lên đầu
    stats = stats[["Feature", "Value", "Count", "Percent"]]

    cate_list.append(stats)

categorical_stats = pd.concat(cate_list, ignore_index=True)

categorical_stats.to_csv(CATE_OUT, index=False)

# =========================
# Summary
# =========================
print(f"Numeric features       : {len(num_features)}")
print(f"Binary flag features   : {len(flag_features)}")
print(f"Categorical features   : {len(cat_features)}")

print(f"\nSaved:")
print(NUMERIC_OUT)
print(CATE_OUT)

Numeric features       : 184
Binary flag features   : 56
Categorical features   : 72

Saved:
D:\KL-lieuvth2\numeric_statistics.csv
D:\KL-lieuvth2\categorical_statistics.csv
